In [13]:
%load_ext autoreload
%autoreload 2

In [14]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [15]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [16]:
# SEED = 0
SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [17]:
import random
import json

random.seed(SEED)

In [18]:
model_type = 'llama'
# model_type = 'qwen'

# MODEL_VERSION = '3'
MODEL_VERSION = '3.1'
# MODEL_VERSION = '3.3'

MODEL_SIZE = '8B'
# MODEL_SIZE = '70B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [19]:
hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))
print(hidden_layers)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]


In [20]:
with open('/scratch/bbjr/skarmakar/neuinv/sk2_items/inter_layer/llama8b/combined_offset1.pkl', 'rb') as file:
# with open('/scratch/bbjr/skarmakar/neuinv/sk2_items/inter_layer/llama8b/combined_all_layers_offset1.pkl', 'rb') as file:
    lrr_models = pickle.load(file)

In [21]:
test_set = [('nocturnal', 'time'), ('offline', 'state'), ('methodical', 'complexity'), ('grueling', 'complexity'), ('first', 'time'), ('relaxed', 'time'), ('gradual', 'physical'), ('shallow', 'physical'), ('matte', 'texture'), ('sophisticated', 'complexity'), ('booming', 'texture'), ('perennial', 'time'), ('relevant', 'logic'), ('saturated', 'texture'), ('set', 'state'), ('disgusting', 'texture'), ('defined', 'texture'), ('unlocked', 'state'), ('friendly', 'social'), ('sloped', 'physical'), ('considerate', 'social'), ('helpful', 'social'), ('dull', 'texture'), ('coarse', 'texture'), ('coherent', 'complexity'), ('just', 'logic'), ('towering', 'physical'), ('sound', 'logic'), ('amused', 'social'), ('salty', 'texture'), ('impartial', 'social'), ('firm', 'texture'), ('dispensable', 'logic'), ('arcane', 'complexity'), ('symmetrical', 'physical'), ('accelerated', 'time'), ('bald', 'texture'), ('despondent', 'social'), ('rank', 'texture'), ('operational', 'state'), ('luminous', 'physical'), ('articulate', 'complexity'), ('charming', 'social'), ('previous', 'time'), ('abrupt', 'time'), ('rushed', 'time'), ('gigantic', 'physical'), ('lumpy', 'texture'), ('unsalted', 'texture'), ('constant', 'time'), ('distant', 'social'), ('meaningful', 'complexity'), ('saturated', 'physical'), ('enduring', 'time'), ('disloyal', 'social'), ('active', 'physical'), ('proven', 'logic'), ('obligatory', 'logic'), ('plugged', 'state'), ('hypothetical', 'logic'), ('subtle', 'complexity'), ('closed', 'state'), ('irrelevant', 'logic'), ('noble', 'social'), ('loose', 'state'), ('murky', 'complexity'), ('persistent', 'time'), ('absolute', 'logic'), ('watery', 'texture'), ('evil', 'social'), ('public', 'complexity'), ('permanent', 'time'), ('cramped', 'physical'), ('loving', 'social'), ('uncomfortable', 'texture'), ('purified', 'state'), ('standard', 'logic'), ('tough', 'physical'), ('forgiving', 'social'), ('occupied', 'state'), ('timeless', 'time'), ('thick', 'physical'), ('passé', 'time'), ('impulsive', 'time'), ('static', 'time'), ('voluminous', 'physical'), ('wicked', 'social'), ('curtailed', 'time'), ('guilty', 'social'), ('stretched', 'physical'), ('adult', 'time'), ('stopped', 'state'), ('kind', 'social'), ('illogical', 'logic'), ('dysfunctional', 'state'), ('icy', 'physical'), ('elaborate', 'complexity'), ('disorderly', 'complexity'), ('animate', 'logic'), ('mellow', 'texture'), ('unfitting', 'logic'), ('there', 'logic'), ('authorized', 'logic'), ('glowing', 'texture'), ('perfumed', 'texture'), ('unknown', 'social'), ('cold', 'texture'), ('dirty', 'physical'), ('primitive', 'time'), ('smug', 'social'), ('gradual', 'time'), ('menopausal', 'time'), ('declining', 'time'), ('rude', 'social'), ('synchronized', 'time'), ('shrunken', 'physical'), ('lucid', 'complexity'), ('unreasonable', 'logic'), ('full', 'physical'), ('peaceful', 'social'), ('creamy', 'texture'), ('uncertain', 'logic'), ('incomplete', 'complexity'), ('exclusive', 'state'), ('bold', 'social'), ('new', 'time'), ('extended', 'time'), ('pregnant', 'state'), ('unbiased', 'logic'), ('thin', 'physical'), ('sharp', 'logic'), ('massive', 'physical'), ('distinct', 'complexity'), ('lawful', 'logic'), ('hostile', 'social'), ('expeditious', 'time'), ('emergent', 'time'), ('unblemished', 'state'), ('agitated', 'social'), ('unlatched', 'state'), ('invalid', 'logic'), ('remote', 'physical'), ('fitful', 'time'), ('far', 'physical'), ('infinite', 'time'), ('speedy', 'time'), ('lifelong', 'time'), ('used', 'time')]

In [22]:
base_prompts = {"complexity": 'Describe the following task, concept, or system. \nTopic: {statement}',
                "logic":'Analyze the following assertion. \nAssertion: {statement}', 
                "physical":'Write a description of the following item. \nItem: {statement}',
                "social":'What are your thoughts on the following statement? \nStatement: {statement}',
                "state":'Describe the current state of the following item. \nItem: {statement}',
                "texture":'Describe the sensory experience of interacting with the following item. \nItem: {statement}',
                "time":'Describe the following event or entity. \nSubject: {statement}',}

In [23]:
categories = ["physical", "texture", "time", "complexity", "logic", "state", "social"]
all_statements = {}

for category in categories:
    all_statements[category] = read_to_list(f"../data/general_statements_adj/{category}/test.txt")
    # all_statements[category] = read_to_list(f"../data/general_statements_adj/{category}/class_1.txt")

In [24]:
def generate_prop_eval_data(test_set, lrr_models, hidden_layers, og_layers=[-31, -25, -20, -15], recursive=True, coef=0.75, max_tokens=100, n=10):
    send_to_eval = {}
    i = 0

    for concept in test_set:
        category = concept[-1]
        c = concept[0]

        category_statements = all_statements[category]
        
        rand_statements = random.sample(category_statements, n)


        prompts = [base_prompts[category].format(statement=s) for s in rand_statements]

        c_controller = load_controller(llm, c, path=f'../all_gitignore/directions_adjectives_llama/{category}/')

        orig_out = test_concept_vector(c_controller, concept=c, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

        # print("Using the combined Matrix.")
        prop_c = propagate_apply_auto(c_controller.directions, lrr_models, hidden_layers, og_layers, recursive=recursive)
        c_controller.directions = prop_c
        
        pred_out = test_concept_vector(c_controller, concept=f"prop {c}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

        for o in range(len(orig_out)):
            u = clean_llama(orig_out[o]).find("-----------------------------------------------------")
            inv_u = clean_llama(orig_out[o]).find("-----------------------------------------------------")

            send_to_eval[i] = {"Target Concept" : c,
                                "User Prompt": prompts[o],
                                "Original Response": clean_llama(orig_out[o])[u+54:],
                                "Predicted Response": clean_llama(pred_out[o])[inv_u+54:],}

            i += 1
            
        # break
    
    return send_to_eval

In [25]:
send_to_eval = generate_prop_eval_data(test_set, lrr_models, hidden_layers=hidden_layers, og_layers=[-31, -25, -20, -15], recursive=True, n=5)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found

========================== + nocturnal Control (normal) ==========================
Describe the following event or entity. 
Subject: The agonizing wait for the test results.
-----------------------------------------------------
The anticipation of the test results has been a long and arduous one, with the medical community holding its collective breath in anticipation of the definitive verdict. The wait for the test results has been a torturous one, with the medical community clinging to the hope that the definitive verdict will finally arrive, bringing an end to the agonizing wait for the test results.

========================== + nocturnal Control (n

In [26]:
filename = f"../all_gitignore/eval_json/prop/prop_matrix_combined/original{SEED}.json"
# filename = "eval_json/prop/prop_matrix_combined_all_layer/original.json"

with open(filename, 'w') as f:
    json.dump(send_to_eval, f, indent=4)

Parser

In [28]:
import json

In [35]:
files = ["1_response.json", "2_response.json", "3_response.json", "4_response.json"]

all_res = {}

for f in files:
    # with open(f'eval_json/prop/original/{f}', 'r', encoding='utf-8') as file:
    # with open(f'eval_json/prop/prop_matrix_combined/{f}', 'r', encoding='utf-8') as file:

    # with open(f'../all_gitignore/eval_json/prop/original/1/{f}', 'r', encoding='utf-8') as file:
    with open(f'../all_gitignore/eval_json/prop/prop_matrix_combined/1/{f}', 'r', encoding='utf-8') as file:

    # with open(f'eval_json/prop/prop_matrix_combined_all_layer/{f}', 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    all_res.update(data)

print(len(all_res))

740


In [36]:
score = 0

for ind in all_res:
    score += all_res[ind]['Evaluation Score']

score = score / len(all_res)
print(score)

7.05


In [ ]:
# Original vectors:                                     7.04054054054054, (1)7.414864864864865
# With the propagation matrix (combined):               8.205405405405406, (1)7.05
# With the propagation matrix (combined + all layers) : 7.462162162162162